# Read-strategy tests for `time=1` chunking (PACE sandbox)

PACE OCI is stored one day per chunk (`time=1`). Reading long time spans from that layout is slow, so
before touching the real PACE store we develop and measure a read strategy on `IO_rechunked.zarr`,
which has the same `time=1` spatial chunking in a small, public, no-auth dataset.

Each section isolates **one lever** and times it against a baseline. We time raw block loads (pure I/O),
so this notebook needs only `xarray` + `dask`, not the model or the standardization step. Numbers will
vary run to run and with filesystem caching; re-run a cell a couple of times and watch the ratios, not
the absolute seconds.

In [1]:
import os
from time import perf_counter
import numpy as np
import xarray as xr
import dask

## Open both layouts

`IO.zarr` is the original (`time=100`, whole spatial domain). `IO_rechunked.zarr` is the PACE-like
layout (`time=1`, `lat=40`, `lon=56`). Same data, same Arabian Sea crop. The printout shows how the
identical region is split into chunks very differently on disk.

In [2]:
lat_min, lat_max = 5, 31
lon_min, lon_max = 42, 80

def open_crop(path):
    ds = xr.open_zarr(path, chunks={})
    return ds.sel(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max))

orig = open_crop(os.path.expanduser("~/shared/mind_the_chl_gap/IO.zarr"))
rech = open_crop("/home/jovyan/shared-public/mindthegap/data/IO_rechunked.zarr")

for name, ds in [("original", orig), ("rechunked", rech)]:
    cs = ds["CHL_cmes-level3"].chunksizes
    print(f"{name:10s} time={tuple(cs['time'])[:3]}...  lat={tuple(cs['lat'])}  lon={tuple(cs['lon'])}")

original   time=(100, 100, 100)...  lat=(105,)  lon=(153,)
rechunked  time=(1, 1, 1)...  lat=(36, 40, 29)  lon=(56, 56, 41)


## Timing helper

`time_load` returns wall-clock seconds to pull a block into memory. It reads only the raw variables
(the actual on-disk reads), under a chosen dask scheduler so we can compare serial vs parallel.

In [3]:
RAW = ["CHL_cmes-level3", "CHL_cmes-cloud", "u_wind", "v_wind", "sst", "air_temp"]

def time_load(ds, tsel, lat=slice(None), lon=slice(None), scheduler="threads"):
    sub = ds.isel(time=tsel, lat=lat, lon=lon)[RAW]
    with dask.config.set(scheduler=scheduler):
        t = perf_counter()
        sub.load()
        return perf_counter() - t

## Baseline: the same 100-day block, two layouts

A 100-day whole-domain block is **one** on-disk chunk in the original file, and ~900 tiny chunks in
the rechunked file (100 days x ~9 spatial tiles). This is the gap we are trying to close.

In [4]:
t_orig = time_load(orig, slice(0, 100))
t_rech = time_load(rech, slice(0, 100))
print(f"original  100-day block: {t_orig:6.2f} s")
print(f"rechunked 100-day block: {t_rech:6.2f} s   ({t_rech / t_orig:.1f}x slower)")

original  100-day block:   0.10 s
rechunked 100-day block:   9.85 s   (102.2x slower)


## Lever 1: parallelize the reads

The rechunked block is hundreds of small reads. `synchronous` does them one at a time; `threads`
issues them concurrently so the per-read latency overlaps. This is the biggest expected lever for
`time=1` data.

In [5]:
t_serial = time_load(rech, slice(0, 100), scheduler="synchronous")
t_thread = time_load(rech, slice(0, 100), scheduler="threads")
print(f"rechunked serial  : {t_serial:6.2f} s")
print(f"rechunked threaded: {t_thread:6.2f} s   ({t_serial / t_thread:.1f}x faster)")

rechunked serial  :  14.83 s
rechunked threaded:   9.96 s   (1.5x faster)


## Lever 2: avoid the time-gather

Loading 100 days at once makes dask assemble 100 one-day chunks into a single block. Loading a few
days at a time keeps reads aligned to how the data is stored: same total reads, but no big re-assembly
and far less memory held at once.

In [6]:
t_100  = time_load(rech, slice(0, 100))
t_10x10 = sum(time_load(rech, slice(k * 10, k * 10 + 10)) for k in range(10))
print(f"one 100-day load : {t_100:6.2f} s")
print(f"ten 10-day loads : {t_10x10:6.2f} s")

one 100-day load :   9.02 s
ten 10-day loads :  21.40 s


## Lever 3: cache a spatial tile in RAM (the pattern that scales)

The scaling-friendly version of "just load everything": hold one spatial tile's time series in memory,
train on all its days, then move to the next tile. Reads are paid once per tile, not once per epoch.
Globally you size the tile so `tile x days` fits in RAM.

(A 40x56 tile is one on-disk spatial chunk on the full grid; the Arab Sea subset offsets it slightly,
so it is roughly, not exactly, one chunk. Persisting the *whole* region also works here because it is
tiny, but that does not scale, which is why we test a single tile.)

In [7]:
tile_lat, tile_lon = slice(0, 40), slice(0, 56)

t_first = time_load(rech, slice(0, 365), lat=tile_lat, lon=tile_lon)   # a year of one tile
print(f"load 1 tile x 365 days (first time): {t_first:6.2f} s")

cached = rech.isel(time=slice(0, 365), lat=tile_lat, lon=tile_lon)[RAW].compute()  # now in RAM
t = perf_counter()
_ = cached.isel(time=200)          # any day, straight from memory
print(f"per-day access from cached tile    : {perf_counter() - t:.5f} s")

load 1 tile x 365 days (first time):   7.47 s
per-day access from cached tile    : 0.00126 s


## Summary

- **Baseline gap:** rechunked `time=1` is Nx slower for a 100-day block (see above).
- **Lever 1 (parallel reads):** expected to recover most of the gap.
- **Lever 2 (avoid the time-gather):** helps memory and skips the re-assembly.
- **Lever 3 (per-tile caching):** amortizes reads across many training steps and is the pattern that
  carries to global PACE.

Whatever combination lands closest to the original-file baseline is the read strategy to bring to the
real PACE store.

In [8]:
import os
print("cores (os.cpu_count):", os.cpu_count())
# the default threaded scheduler used roughly this many workers, which is what the baseline ran with


cores (os.cpu_count): 4


In [9]:
def time_load_nw(ds, tsel, num_workers, lat=slice(None), lon=slice(None)):
    sub = ds.isel(time=tsel, lat=lat, lon=lon)[RAW]
    t = perf_counter()
    dask.compute(sub, scheduler="threads", num_workers=num_workers)
    return perf_counter() - t

# reads are latency-bound, so more workers than cores can still help
for nw in [1, 2, 4, 8, 16, 32, 64]:
    print(f"num_workers={nw:3d}: {time_load_nw(rech, slice(0, 100), nw):6.2f} s")


num_workers=  1:  16.28 s
num_workers=  2:  10.40 s
num_workers=  4:   8.98 s
num_workers=  8:   9.02 s
num_workers= 16:   8.57 s
num_workers= 32:   7.65 s
num_workers= 64:   9.50 s


In [10]:
EPOCHS = 3
tsel = slice(0, 100)

t = perf_counter()
for _ in range(EPOCHS):
    rech.isel(time=tsel)[RAW].load()               # re-read from disk each epoch (current behavior)
print(f"re-read every epoch ({EPOCHS}x): {perf_counter() - t:6.2f} s")

t = perf_counter()
cached = rech.isel(time=tsel)[RAW].load()          # read once
for _ in range(EPOCHS):
    _ = {v: cached[v].values for v in RAW}         # reuse from RAM
print(f"read once, reuse    ({EPOCHS}x): {perf_counter() - t:6.2f} s")


re-read every epoch (3x):  27.54 s
read once, reuse    (3x):   9.71 s


In [11]:
import threading, queue
import time as _time

blocks = [slice(k * 100, k * 100 + 100) for k in range(3)]
TRAIN_SECONDS = 5   # stand-in for how long the GPU trains on one block

# sequential: load, then "train", repeat
t = perf_counter()
for tsel in blocks:
    rech.isel(time=tsel)[RAW].load()
    _time.sleep(TRAIN_SECONDS)
print(f"sequential (load then train): {perf_counter() - t:5.1f} s")

# prefetched: a background thread loads the next block while we "train"
def _loader(q):
    for tsel in blocks:
        q.put(rech.isel(time=tsel)[RAW].load())
    q.put(None)

t = perf_counter()
q = queue.Queue(maxsize=2)
threading.Thread(target=_loader, args=(q,), daemon=True).start()
while True:
    blk = q.get()
    if blk is None:
        break
    _time.sleep(TRAIN_SECONDS)   # "train" on this block while the loader prefetches the next
print(f"prefetched (load overlaps train): {perf_counter() - t:5.1f} s")


sequential (load then train):  43.4 s
prefetched (load overlaps train):  30.9 s
